# B2-024-gpu-scientific-ml-capstone — Practice p17 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** semi-supervised-pseudo-labeling

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: optional-colab-l4` · seed `20261022` · the CPU correctness path below is required and is what CI runs; the `Accelerator extension` section is optional, runs only on a Colab L4, and is never executed by CI  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

MARGIN = 0.03
CPU_CONFIG = {"device": "cpu", "amp_dtype": None, "thresholds": (0.7, 0.9), "round_caps": (100, 200, 200),
              "steps_per_round": 150, "batch_size": None, "student_noise": 0.3, "extra_pool": 0}

## Solution

**Pieces.**

* `NoisyStudent` wraps the committed `ShapeCNN`; noise is added inside `forward` only in training mode, so the teacher's pseudo-labelling pass (eval mode) sees clean pool images.
* `train_student` is the committed baseline recipe (seed, Adam lr 0.01, 150 full-batch steps under `CPU_CONFIG`), with the student's noise level as the only extra knob; with `noise_std = 0.0` on `(xl, yl)` it reproduces the baseline model exactly. Device, precision, step count, batch size, thresholds, caps, and pool size all come from the config dict, so the L4 run changes only the dictionary.
* The **seam** is a forward pre-hook registered on each new student. Pre-hooks run before `forward`, so they see the clean rows before the noise is added; the hook records the split of every row on the first training-mode call of each `train_student`, then disarms itself.
* `select_pseudo_labels` is the p09 rule (arg-max with lower-class ties, `>=` threshold, per-class cap with lower-index ties, ascending output).

**Self-training.** For each $\tau\in\{0.7,0.9\}$ the teacher starts as the baseline, and three rounds with caps $(100,200,200)$ each pseudo-label the clean pool and retrain a re-initialized noisy student on labelled + pseudo-labelled images. Every choice ($\tau$) uses only the 30 labelled validation images.

In [ ]:
from contextlib import nullcontext

CONFIG = CPU_CONFIG                    # the accelerator extension replaces only this dict
THRESHOLDS = CONFIG["thresholds"]

xl, yl = capstone_data.train_rows("shapes_ssl")
xv, yv = capstone_data.val_rows("shapes_ssl")
pool = capstone_data.unlabelled_rows()
if CONFIG["extra_pool"] > 0:
    extra, _ = capstone_data.simulate("shapes_ssl", CONFIG["extra_pool"], seed=2)      # labels discarded
    pool = torch.cat([pool, extra])
SEAM = []


class NoisyStudent(nn.Module):
    def __init__(self, noise_std):
        super().__init__()
        self.noise_std = noise_std
        self.cnn = capstone_data.ShapeCNN()

    def forward(self, x):
        if self.training and self.noise_std > 0:
            x = x + self.noise_std * torch.randn_like(x)
        return self.cnn(x)


def make_seam_hook():
    state = {"armed": True}

    def hook(module, args):
        if module.training and state["armed"]:
            state["armed"] = False
            SEAM.extend(capstone_data.split_of("shapes_ssl", row) for row in args[0].detach().cpu())
    return hook


def train_student(x, y, noise_std, config):
    device = torch.device(config["device"])
    torch.manual_seed(SEED)
    model = NoisyStudent(noise_std).to(device)
    model.register_forward_pre_hook(make_seam_hook())
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
    model.train()
    x, y = x.to(device), y.to(device)
    amp = config["amp_dtype"]
    for _ in range(config["steps_per_round"]):
        if config["batch_size"] is None:
            xb, yb = x, y
        else:
            idx = torch.randint(0, x.shape[0], (config["batch_size"],))
            xb, yb = x[idx], y[idx]
        optimizer.zero_grad(set_to_none=True)
        with nullcontext() if amp is None else torch.autocast(device_type=device.type, dtype=amp):
            loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        optimizer.step()
    return model.eval()


def select_pseudo_labels(probs, threshold, max_per_class):
    pred = probs.argmax(dim=1)
    conf = probs.max(dim=1).values
    candidate = conf >= threshold
    kept = []
    for k in range(probs.shape[1]):
        rows = torch.nonzero(candidate & (pred == k)).flatten()
        rows = rows[torch.sort(-conf[rows], stable=True).indices]
        kept.append(rows[:max_per_class])
    indices = torch.sort(torch.cat(kept)).values.to(torch.int64)
    return indices, pred[indices].to(torch.int64)


def val_accuracy(model):
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        return (model(xv.to(device)).argmax(dim=1).cpu() == yv).double().mean().item()


TRACE = {}
STUDENTS = {}
for tau in THRESHOLDS:
    teacher = train_student(xl, yl, 0.0, CONFIG)
    rounds = []
    for cap in CONFIG["round_caps"]:
        teacher.eval()
        with torch.no_grad():
            probs = F.softmax(teacher(pool.to(next(teacher.parameters()).device)), dim=1).cpu()
        idx, labels = select_pseudo_labels(probs, tau, cap)
        counts = torch.bincount(labels, minlength=3).tolist()
        teacher = train_student(torch.cat([xl, pool[idx]]), torch.cat([yl, labels]), CONFIG["student_noise"], CONFIG)
        rounds.append({"cap": cap, "n_selected": len(idx), "counts": counts, "val_acc": val_accuracy(teacher)})
    TRACE[tau] = rounds
    STUDENTS[tau] = teacher

baseline_model = train_student(xl, yl, 0.0, CONFIG)
noise_only_model = train_student(xl, yl, CONFIG["student_noise"], CONFIG)
ABLATIONS = {"baseline (labelled only, no noise)": val_accuracy(baseline_model),
             "noise only (labelled only, noise 0.3)": val_accuracy(noise_only_model)}

final_val = {tau: TRACE[tau][-1]["val_acc"] for tau in THRESHOLDS}
best = max(final_val.values())
chosen_tau = max(tau for tau in THRESHOLDS if final_val[tau] == best)       # ties -> larger tau
chosen = STUDENTS[chosen_tau]

for tau in THRESHOLDS:
    for i, r in enumerate(TRACE[tau], 1):
        print(f"tau={tau} round {i}: cap {r['cap']}, selected {r['n_selected']}, class counts {r['counts']}, val acc {r['val_acc']:.4f}")
for name, acc in ABLATIONS.items():
    print(f"{name}: val acc {acc:.4f}")
print("chosen tau:", chosen_tau, "| seam values:", sorted(set(SEAM)))

**Locked test score (one call, after all fitting and selection).**

In [ ]:
baseline = capstone_data.baseline_score("shapes_ssl")
chosen.eval()


def predict_fn(images):
    with torch.no_grad():
        return chosen(images.to(next(chosen.parameters()).device)).argmax(dim=1).cpu().to(torch.int64)


assert capstone_data.test_call_count("shapes_ssl") == 0
test = capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="shapes_ssl")
assert capstone_data.test_call_count("shapes_ssl") == 1
print(f"noisy student (tau={chosen_tau}): test accuracy {test.score:.4f}, 95% CI [{test.ci_low:.4f}, {test.ci_high:.4f}]")
print(f"committed baseline: {baseline.score:.4f} [{baseline.ci_low:.4f}, {baseline.ci_high:.4f}]; bar {baseline.score + MARGIN:.4f}")

**Interpretation (confirmation bias).**
On validation the baseline scores $0.600$ and noise alone $0.733$, while the selected noisy student ($\tau=0.9$, round 3) reaches $0.867$, so part of the gain is regularization from the noise but the pseudo-labelled pool adds about four more correct validation images on top of it.
With $\tau=0.7$ the teacher admits more images (572 by round 3, nearly the whole pool) and validation accuracy stays flat at $0.767$, consistent with confirmation bias: lower-confidence pseudo-labels carry more of the teacher's mistakes, and the student learns them back.
The per-class caps hold every count at or below its cap ($100$, then $\le200$), and the counts stay roughly balanced (e.g. $177/151/155$ at $\tau=0.9$), so no class floods the training set; the slightly lower class-1 and class-2 counts show where the teacher is least confident.
Re-initialized, noisy students cannot simply copy the teacher's exact decisions, which is why accuracy rises across rounds at $\tau=0.9$ instead of collapsing.
With only 30 validation images (one image $=0.033$) these validation gaps are a few images each, so the claim rests on the single locked test score, which clears the $0.84$ bar.

## Accelerator extension

*Optional; Colab L4 only; never executed by CI and never required for credit.*

On a Colab L4, the same loop scales up by passing `L4_CONFIG` in place of `CPU_CONFIG` to the same `train_student` / self-training code. It has exactly the keys of `CPU_CONFIG`, so only the dictionary changes: `batch_size = 1024` switches on mini-batch sampling with the global generator, and `extra_pool = 50000` enlarges the pool with images simulated from the same process, whose labels are discarded.

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "thresholds": (0.7, 0.8, 0.9), "round_caps": (1000, 5000, 10000, 20000),
             "steps_per_round": 2000, "batch_size": 1024, "student_noise": 0.3, "extra_pool": 50000}
```

**What to observe.** Validation accuracy per round as the pool grows (does it keep rising, or does confirmation bias set in?); per-class pseudo-label counts; and the noise-only ablation at the larger scale.

*Documentation only: this section is not executed by CI; the CPU configuration above is the certified path.*

### Answer check

In [ ]:
assert chosen_tau in THRESHOLDS
assert final_val[chosen_tau] == max(final_val.values())
for tau in THRESHOLDS:
    for r in TRACE[tau]:
        assert all(c <= r["cap"] for c in r["counts"])
        assert r["n_selected"] <= 3 * r["cap"] and sum(r["counts"]) == r["n_selected"]
assert len(SEAM) > 0 and set(SEAM) <= {"train", "unlabelled"}
assert capstone_data.test_call_count("shapes_ssl") == 1
assert test.score >= baseline.score + MARGIN